# 097 — Datos sintéticos: utilidad y contaminación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Datos sintéticos**: ejemplos producidos por un modelo generativo o un procedimiento
estadístico, no recolectados del fenómeno real. Usos legítimos: **aumento** de datos,
sustitutos con fines de **privacidad** (sin garantías formales, sintético ≠ anónimo) y
**balanceo de clases** — el clásico **SMOTE** interpola entre una instancia minoritaria
y un vecino: `x_nuevo = x_i + λ(x_vecino − x_i)`, λ ~ U(0,1).

**Utilidad — TSTR** (*Train on Synthetic, Test on Real*): entrena con sintético,
entrena un baseline con real, evalúa AMBOS sobre un test real nunca visto.
`utilidad ≈ métrica(M_sintético) / métrica(M_real)`, idealmente cercana a 1.

**Model collapse** (Shumailov et al., Nature 2024): al entrenar generaciones sucesivas
sobre salidas de la generación anterior, el muestreo finito pierde las colas de la
distribución (colapso temprano) y la varianza se contrae hasta una distribución
estrecha (colapso tardío). La pérdida de una cola es un **estado absorbente**: si la
clase rara no se muestrea una vez, ninguna generación posterior la recupera.

## 🧮 Ejemplo de referencia

Clase `rara` con p = 0.05, muestreo de N = 20 por generación:
`P(0 raros en una generación) = 0.95²⁰ ≈ 0.358`. La probabilidad de que la cola
sobreviva 3 generaciones (aproximando p̂ ≈ 0.05 mientras viva) es
`(1 − 0.358)³ ≈ 0.265`: en ~3 de cada 4 corridas, tras tres generaciones recursivas
el modelo asigna probabilidad 0 a un evento que ocurre el 5 % de las veces.
Verifica estos números a mano antes de ejecutar el laboratorio.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("evaluation", seed=97)
show(result)


## Reflexión

1. Con p_rara = 0.05 y N = 20, el riesgo de perder la cola en una generación es 0.95²⁰ ≈ 0.36. ¿Cuánto vale con N = 200 y qué implica sobre el tamaño de muestra en pipelines recursivos?
2. ¿Por qué TSTR mide utilidad mejor que una métrica de fidelidad marginal (p. ej. parecido visual), y qué relación captura que la fidelidad no captura?
3. Si en un crawl web no puedes distinguir el contenido generado, ¿qué aporta la procedencia activa (clase 098) como mitigación del collapse y cuál es su límite (¿quién marca y quién no)?